In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

In [2]:
# ── Configuration ──
np.random.seed(42)  # Reproducibility
NUM_PROFILES = 50000
OUTPUT_PATH = Path("data/synthetic/credit_scoring_dataset.csv")
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

print(f"🧪 Generating {NUM_PROFILES:,} synthetic credit profiles...")


🧪 Generating 50,000 synthetic credit profiles...


In [3]:
# ── Base Demographics ──
customer_ids = [f"CUST-{i+1:06d}" for i in range(NUM_PROFILES)]

gender = np.random.choice(["Male", "Female", "Other"], size=NUM_PROFILES, p=[0.48, 0.48, 0.04])

occupation = np.random.choice(
    ["Professional", "Skilled", "Clerical", "Self-Employed", "Unemployed"],
    size=NUM_PROFILES,
    p=[0.30, 0.25, 0.20, 0.15, 0.10]
)

In [4]:
# Salary correlates with occupation
salary_map = {
    "Professional":    (65000, 12000),
    "Skilled":        (38000,  8000),
    "Clerical":       (26000,  6000),
    "Self-Employed":  (42000, 15000),
    "Unemployed":     (10000,  3000)
}
annual_salary = np.array([np.random.normal(*salary_map[occ]) for occ in occupation]).clip(8000, 95000).round(0)

# ── Credit Card Usage ──
credit_card_utilization = np.random.beta(2.2, 3.5, NUM_PROFILES) * 100  # Skewed low → good


In [5]:
# ── Payment Behaviour ──
# Correlate on-time payments with salary & utilization
base_payment = 0.65 + (annual_salary / 100000) * 0.20 - (credit_card_utilization / 200) * 0.30
on_time_payment_ratio = np.clip(base_payment + np.random.normal(0, 0.08, NUM_PROFILES), 0.30, 1.00).round(4) * 100

# ── Mortgage ──
mortgage_probs = {"Professional": 0.55, "Skilled": 0.35, "Clerical": 0.20, "Self-Employed": 0.25, "Unemployed": 0.02}
mortgage = np.array([
    np.random.choice(
        ["None", "Current", "Arrears", "PaidOff"],
        p=[
            1 - mortgage_probs[occ],
            mortgage_probs[occ] * 0.75,
            mortgage_probs[occ] * 0.10,
            mortgage_probs[occ] * 0.15
        ]
    ) for occ in occupation
])

In [6]:
# ── Rent ──
rent_monthly = np.where(
    mortgage == "None",
    np.random.lognormal(5.5, 0.35, NUM_PROFILES).round(-1).clip(300, 2500),
    0.0
)
rent_on_time_rate = np.where(
    mortgage == "None",
    np.clip(on_time_payment_ratio + np.random.normal(5, 8, NUM_PROFILES), 40, 100).round(1),
    100.0  # Not renting → neutral
)

# ── Utility Bills ──
utility_bills_on_time = np.clip(on_time_payment_ratio + np.random.normal(3, 10, NUM_PROFILES), 25, 100).round(1)

# ── Mobile Money ──
momo_active = np.random.choice([True, False], size=NUM_PROFILES, p=[0.65, 0.35])
momo_tx_monthly = np.where(momo_active, np.random.poisson(12, NUM_PROFILES).clip(0, 120), 0)
momo_age_days = np.where(momo_active, np.random.randint(30, 1825, NUM_PROFILES), 0)

# ── Loan Term ──
loan_term_months = np.random.choice([6, 12, 24, 36, 60], size=NUM_PROFILES, p=[0.15, 0.25, 0.30, 0.20, 0.10])

# ── Additional Credit Features ──
existing_loans_count = np.random.poisson(0.8, NUM_PROFILES).clip(0, 5)
months_credit_history = np.random.exponential(80, NUM_PROFILES).clip(0, 360).astype(int)

# ── TARGET: Default Probability & Label ──
# Formula: higher utilization + late payments + arrears → much higher default risk
default_risk = (
    0.40 * (credit_card_utilization / 100) +
    0.45 * (1 - on_time_payment_ratio / 100) +
    0.30 * np.where(mortgage == "Arrears", 0.8, np.where(mortgage == "Current", 0.1, 0.0)) +
    0.15 * np.minimum(existing_loans_count / 3, 1.0) -
    0.20 * np.minimum(months_credit_history / 120, 1.0) -
    0.10 * np.minimum(annual_salary / 60000, 1.0) +
    np.random.normal(0, 0.06, NUM_PROFILES)
)
default_risk = np.clip(default_risk, 0.01, 0.75)
default_next_12m = np.random.binomial(1, default_risk, NUM_PROFILES)

# ── Build DataFrame ──
df = pd.DataFrame({
    "customer_id": customer_ids,
    "gender": gender,
    "occupation": occupation,
    "annual_salary_gbp": annual_salary.astype(int),
    "credit_card_utilization_pct": credit_card_utilization.round(1),
    "on_time_payment_ratio_pct": on_time_payment_ratio.round(1),
    "mortgage_status": mortgage,
    "rent_payment_monthly_gbp": rent_monthly.astype(int),
    "rent_on_time_rate_pct": rent_on_time_rate,
    "utility_bills_on_time_pct": utility_bills_on_time,
    "mobile_money_active": momo_active,
    "mobile_money_tx_monthly": momo_tx_monthly,
    "mobile_money_age_days": momo_age_days,
    "preferred_loan_term_months": loan_term_months,
    "existing_loans_count": existing_loans_count,
    "months_credit_history": months_credit_history,
    "default_next_12m": default_next_12m
})

# ── Save ──
df.to_csv(OUTPUT_PATH, index=False)

# ── Print Summary ──
print(f"\n✅ DATASET SAVED → {OUTPUT_PATH}")
#print(f"📊 Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
#print(f"\n📈 DEFAULT RATE: {df['default_next_12m'].mean():.1%}")
#print(f"\n💰 Salary Distribution:")
#print(f"   Mean: £{df['annual_salary_gbp'].mean():,.0f}  |  Median: £{df['annual_salary_gbp'].median():,.0f}")
#print(f"\n💳 Credit Card Utilization: {df['credit_card_utilization_pct'].mean():.1f}% (avg)")
#print(f"📅 On-Time Payments: {df['on_time_payment_ratio_pct'].mean():.1f}% (avg)")
#print(f"\n🏠 Mortgage Status:")
#for status, pct in df["mortgage_status"].value_counts(normalize=True).items():print(f"   {status:12} {pct:.1%}")
#print(f"\n📱 Mobile Money Users: {df['mobile_money_active'].mean():.1%}")
#print(f"\n⚖️  Fairness Check — Default Rate by Gender:")
#for g in sorted(df["gender"].unique()):
    #rate = df[df["gender"]==g]["default_next_12m"].mean()
    #print(f"   {g:10} {rate:.1%}")


✅ DATASET SAVED → data\synthetic\credit_scoring_dataset.csv


In [7]:
df.head()


,customer_id,gender,occupation,annual_salary_gbp,credit_card_utilization_pct,on_time_payment_ratio_pct,mortgage_status,rent_payment_monthly_gbp,rent_on_time_rate_pct,utility_bills_on_time_pct,mobile_money_active,mobile_money_tx_monthly,mobile_money_age_days,preferred_loan_term_months,existing_loans_count,months_credit_history,default_next_12m
0,CUST-000001,Male,Self-Employed,54641,24.3,71.9,None,320,68.4,87.9,False,0,0,6,1,20,0
1,CUST-000002,Female,Skilled,58191,9.5,71.1,None,470,76.1,79.2,True,13,1208,60,1,41,0
2,CUST-000003,Female,Professional,63798,50.3,66.4,Current,0,100.0,78.7,False,0,0,36,0,208,0
3,CUST-000004,Female,Clerical,12814,32.4,60.9,None,300,71.0,68.2,False,0,0,60,2,35,0
4,CUST-000005,Male,Skilled,49403,7.3,81.9,None,300,79.8,79.8,True,9,1818,6,0,15,0


In [8]:
df.shape


(50000, 17)

In [10]:
df.head()


,customer_id,gender,occupation,annual_salary_gbp,credit_card_utilization_pct,on_time_payment_ratio_pct,mortgage_status,rent_payment_monthly_gbp,rent_on_time_rate_pct,utility_bills_on_time_pct,mobile_money_active,mobile_money_tx_monthly,mobile_money_age_days,preferred_loan_term_months,existing_loans_count,months_credit_history,default_next_12m
0,CUST-000001,Male,Self-Employed,54641,24.3,71.9,None,320,68.4,87.9,False,0,0,6,1,20,0
1,CUST-000002,Female,Skilled,58191,9.5,71.1,None,470,76.1,79.2,True,13,1208,60,1,41,0
2,CUST-000003,Female,Professional,63798,50.3,66.4,Current,0,100.0,78.7,False,0,0,36,0,208,0
3,CUST-000004,Female,Clerical,12814,32.4,60.9,None,300,71.0,68.2,False,0,0,60,2,35,0
4,CUST-000005,Male,Skilled,49403,7.3,81.9,None,300,79.8,79.8,True,9,1818,6,0,15,0


In [14]:
X = df.drop(["customer_id", "default_next_12m"], axis=1)
y = df["default_next_12m"]


In [15]:
X

,gender,occupation,annual_salary_gbp,credit_card_utilization_pct,on_time_payment_ratio_pct,mortgage_status,rent_payment_monthly_gbp,rent_on_time_rate_pct,utility_bills_on_time_pct,mobile_money_active,mobile_money_tx_monthly,mobile_money_age_days,preferred_loan_term_months,existing_loans_count,months_credit_history
0,Male,Self-Employed,54641,24.3,71.9,None,320,68.4,87.9,False,0,0,6,1,20
1,Female,Skilled,58191,9.5,71.1,None,470,76.1,79.2,True,13,1208,60,1,41
2,Female,Professional,63798,50.3,66.4,Current,0,100.0,78.7,False,0,0,36,0,208
3,Female,Clerical,12814,32.4,60.9,None,300,71.0,68.2,False,0,0,60,2,35
4,Male,Skilled,49403,7.3,81.9,None,300,79.8,79.8,True,9,1818,6,0,15
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49995,Female,Self-Employed,26533,19.6,51.8,None,300,56.4,45.3,True,11,1622,24,1,11
49996,Male,Self-Employed,64644,19.8,71.5,None,300,61.5,73.7,False,0,0,24,2,94
49997,Female,Clerical,20625,31.6,37.4,None,300,43.4,25.0,True,10,666,6,2,13
49998,Female,Skilled,34883,29.1,57.9,None,430,64.0,69.2,True,9,1423,36,0,113


In [16]:
y

0        0
1        0
2        0
3        0
4        0
        ..
49995    0
49996    0
49997    0
49998    0
49999    0
Name: default_next_12m, Length: 50000, dtype: int32

✅ KEY FEATURES OF THIS DATASET
✅ Realistic Correlations — higher income → lower default; high utilization → higher default
✅ Balanced Labels — ~15–20% default rate (matches real-world)
✅ Demographic Included — built-in fairness audit capability
✅ Mixed Data Types — numeric + categorical → tests your full pipeline
✅ Noisy & Imperfect — mimics real-world data quality
✅ Reproducible — fixed seed → same dataset every run
✅ Ready for Supervised Learning — clear default_next_12m target column

In [17]:
"""Generate small 1K test set for fast testing"""

import numpy as np
import pandas as pd
from pathlib import Path

np.random.seed(123)
NUM = 1000
OUT = Path("data/synthetic/test_set_1000.csv")
OUT.parent.mkdir(parents=True, exist_ok=True)

# Reuse same realistic distributions from main generator
ids = [f"TEST-{i+1:04d}" for i in range(NUM)]
gender = np.random.choice(["Male", "Female", "Other"], NUM, p=[0.48, 0.48, 0.04])
occupation = np.random.choice(
    ["Professional", "Skilled", "Clerical", "Self-Employed", "Unemployed"],
    NUM, p=[0.30, 0.25, 0.20, 0.15, 0.10]
)

salary_map = {"Professional":(65000,12000),"Skilled":(38000,8000),"Clerical":(26000,6000),"Self-Employed":(42000,15000),"Unemployed":(10000,3000)}
annual_salary = np.array([np.random.normal(*salary_map[o]) for o in occupation]).clip(8000,95000).round(0)
cc_util = np.random.beta(2.2,3.5,NUM)*100
base_pay = 0.65 + (annual_salary/100000)*0.20 - (cc_util/200)*0.30
on_time = np.clip(base_pay + np.random.normal(0,0.08,NUM),0.30,1.00).round(4)*100

mort_prob = {"Professional":0.55,"Skilled":0.35,"Clerical":0.20,"Self-Employed":0.25,"Unemployed":0.02}
mortgage = np.array([np.random.choice(["None","Current","Arrears","PaidOff"],
    p=[1-mort_prob[o], mort_prob[o]*0.75, mort_prob[o]*0.10, mort_prob[o]*0.15]) for o in occupation])

rent = np.where(mortgage=="None", np.random.lognormal(5.5,0.35,NUM).round(-1).clip(300,2500), 0.0)
rent_ot = np.where(mortgage=="None", np.clip(on_time+np.random.normal(5,8,NUM),40,100).round(1), 100.0)
util_ot = np.clip(on_time+np.random.normal(3,10,NUM),25,100).round(1)
momo_active = np.random.choice([True,False],NUM,p=[0.65,0.35])
momo_tx = np.where(momo_active, np.random.poisson(12,NUM).clip(0,120),0)
momo_age = np.where(momo_active, np.random.randint(30,1825,NUM),0)
term = np.random.choice([6,12,24,36,60],NUM,p=[0.15,0.25,0.30,0.20,0.10])
loans = np.random.poisson(0.8,NUM).clip(0,5)
history = np.random.exponential(80,NUM).clip(0,360).astype(int)

dr = 0.40*(cc_util/100) + 0.45*(1-on_time/100) + 0.30*np.where(mortgage=="Arrears",0.8,np.where(mortgage=="Current",0.1,0.0)) + 0.15*np.minimum(loans/3,1.0) - 0.20*np.minimum(history/120,1.0) - 0.10*np.minimum(annual_salary/60000,1.0)
dr = np.clip(dr+np.random.normal(0,0.06,NUM),0.01,0.75)
default = np.random.binomial(1,dr,NUM)

pd.DataFrame({
    "customer_id":ids,"gender":gender,"occupation":occupation,
    "annual_salary_gbp":annual_salary.astype(int),"credit_card_utilization_pct":cc_util.round(1),
    "on_time_payment_ratio_pct":on_time.round(1),"mortgage_status":mortgage,
    "rent_payment_monthly_gbp":rent.astype(int),"rent_on_time_rate_pct":rent_ot,
    "utility_bills_on_time_pct":util_ot,"mobile_money_active":momo_active,
    "mobile_money_tx_monthly":momo_tx,"mobile_money_age_days":momo_age,
    "preferred_loan_term_months":term,"existing_loans_count":loans,
    "months_credit_history":history,"default_next_12m":default
}).to_csv(OUT,index=False)

print(f"✅ Test set saved → {OUT}")
print(f"📊 Default Rate: {default.mean():.1%}")

✅ Test set saved → data\synthetic\test_set_1000.csv
📊 Default Rate: 18.1%


In [19]:
#SOFTINT AI — Credit Scoring Model Training Pipeline
#Loads CSV → Encodes features → Trains 2 models → Evaluates → Saves best model
#"""

import pandas as pd
import numpy as np
import pickle
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score, f1_score, confusion_matrix, classification_report

# ── Configuration ──
DATA_PATH = Path("data/synthetic/credit_scoring_dataset.csv")
TEST_DATA_PATH = Path("data/synthetic/test_set_1000.csv")
MODEL_DIR = Path("models")
MODEL_DIR.mkdir(exist_ok=True)

TARGET = "default_next_12m"
DROP_COLS = ["customer_id"]

# ── Feature Types ──
NUMERIC = [
    "annual_salary_gbp", "credit_card_utilization_pct", "on_time_payment_ratio_pct",
    "rent_payment_monthly_gbp", "rent_on_time_rate_pct", "utility_bills_on_time_pct",
    "mobile_money_tx_monthly", "mobile_money_age_days", "preferred_loan_term_months",
    "existing_loans_count", "months_credit_history"
]
CATEGORICAL = ["gender", "occupation", "mortgage_status", "mobile_money_active"]

# ── Load Data ──
def load_data(path):
    if not path.exists():
        raise FileNotFoundError(f"Dataset not found: {path}\n→ Run the generator script first!")
    df = pd.read_csv(path)
    print(f"✅ Loaded {len(df):,} profiles from {path.name}")
    return df

# ── Build Preprocessor ──
def build_preprocessor():
    return ColumnTransformer(
        transformers=[
            ("num", StandardScaler(), NUMERIC),
            ("cat", OneHotEncoder(drop="first", sparse_output=False, handle_unknown="ignore"), CATEGORICAL)
        ])

# ── Train & Evaluate ──
def train_and_evaluate(X_train, X_test, y_train, y_test, model, name):
    print(f"\n{'='*50}")
    print(f"🤖 MODEL: {name}")
    print(f"{'='*50}")

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    # Metrics
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    roc = roc_auc_score(y_test, y_proba)
    f1 = f1_score(y_test, y_pred)

    print(f"📊 Accuracy:  {acc:.1%}")
    print(f"📊 Precision: {prec:.1%}")
    print(f"📊 Recall:    {rec:.1%}")
    print(f"📊 F1 Score:  {f1:.1%}")
    print(f"📊 AUC-ROC:   {roc:.3f}")
    print(f"\n📋 Classification Report:\n{classification_report(y_test, y_pred, zero_division=0)}")
    print(f"🔍 Confusion Matrix:\n{confusion_matrix(y_test, y_pred)}")

    return model, roc

# ── Fairness Check ──
def fairness_check(df_test, preds):
    print(f"\n{'='*50}")
    print(f"⚖️  FAIRNESS CHECK — Default Rate by Gender")
    print(f"{'='*50}")
    df_test["predicted"] = preds
    for g in sorted(df_test["gender"].unique()):
        subset = df_test[df_test["gender"] == g]
        rate = subset["predicted"].mean()
        actual = subset[TARGET].mean()
        print(f"   {g:10} Predicted Default: {rate:.1%}  |  Actual: {actual:.1%}")
    print(f"   ✅ Gap between groups should be <5% for fairness compliance")

# ── Predict New Applicant Example ──
def score_new_applicant(pipeline, applicant):
    """Score a single new applicant dictionary"""
    df = pd.DataFrame([applicant])
    proba = pipeline.predict_proba(df)[:, 1][0]
    score = int(850 - (proba * 550))  # Map 0-1 → 300-850 scale
    grade = "A" if score>=750 else "B" if score>=650 else "C" if score>=550 else "D" if score>=400 else "E"
    return {"score_300_850": score, "grade": grade, "default_risk_pct": round(proba*100,1)}

# ── MAIN EXECUTION ──
if __name__ == "__main__":
    print("🚀 SOFTINT AI — Credit Scoring Model Training Pipeline\n")

    # Load main dataset
    df = load_data(DATA_PATH)
    X = df.drop([TARGET]+DROP_COLS, axis=1)
    y = df[TARGET]

    # Split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    df_test = df.loc[X_test.index].copy()
    print(f"📊 Train: {len(X_train):,} | Test: {len(X_test):,} | Default Rate: {y.mean():.1%}")

    # Preprocessor
    pre = build_preprocessor()

    # Model 1 — Logistic Regression
    lr_pipe = Pipeline(steps=[("pre", pre), ("model", LogisticRegression(max_iter=1000))])
    lr_model, lr_roc = train_and_evaluate(X_train, X_test, y_train, y_test, lr_pipe, "Logistic Regression")

    # Model 2 — Random Forest
    rf_pipe = Pipeline(steps=[("pre", pre), ("model", RandomForestClassifier(n_estimators=100, random_state=42))])
    rf_model, rf_roc = train_and_evaluate(X_train, X_test, y_train, y_test, rf_pipe, "Random Forest")

    # Fairness on best model
    best_model = rf_model if rf_roc >= lr_roc else lr_model
    fairness_check(df_test, best_model.predict(X_test))

    # Save best model
    model_path = MODEL_DIR / "credit_scoring_model.pkl"
    with open(model_path, "wb") as f:
        pickle.dump(best_model, f)
    print(f"\n💾 Best model saved → {model_path}")

    # ── Example: Score a new applicant ──
    print(f"\n{'='*50}")
    print(f"🧪 EXAMPLE — Score New Applicant")
    print(f"{'='*50}")
    applicant = {
        "gender": "Female",
        "occupation": "Professional",
        "annual_salary_gbp": 58000,
        "credit_card_utilization_pct": 28.5,
        "on_time_payment_ratio_pct": 96.0,
        "mortgage_status": "Current",
        "rent_payment_monthly_gbp": 0,
        "rent_on_time_rate_pct": 100.0,
        "utility_bills_on_time_pct": 98.0,
        "mobile_money_active": True,
        "mobile_money_tx_monthly": 15,
        "mobile_money_age_days": 720,
        "preferred_loan_term_months": 24,
        "existing_loans_count": 1,
        "months_credit_history": 48
    }
    result = score_new_applicant(best_model, applicant)
    print(f"   Credit Score: {result['score_300_850']} / 850")
    print(f"   Grade:        {result['grade']}")
    print(f"   Risk:         {result['default_risk_pct']}% probability of default")

🚀 SOFTINT AI — Credit Scoring Model Training Pipeline

✅ Loaded 50,000 profiles from credit_scoring_dataset.csv
📊 Train: 40,000 | Test: 10,000 | Default Rate: 19.5%

🤖 MODEL: Logistic Regression
📊 Accuracy:  80.4%
📊 Precision: 46.7%
📊 Recall:    6.2%
📊 F1 Score:  10.9%
📊 AUC-ROC:   0.719

📋 Classification Report:
              precision    recall  f1-score   support

           0       0.81      0.98      0.89      8054
           1       0.47      0.06      0.11      1946

    accuracy                           0.80     10000
   macro avg       0.64      0.52      0.50     10000
weighted avg       0.75      0.80      0.74     10000

🔍 Confusion Matrix:
[[7917  137]
 [1826  120]]

🤖 MODEL: Random Forest
📊 Accuracy:  80.2%
📊 Precision: 40.4%
📊 Recall:    3.4%
📊 F1 Score:  6.3%
📊 AUC-ROC:   0.697

📋 Classification Report:
              precision    recall  f1-score   support

           0       0.81      0.99      0.89      8054
           1       0.40      0.03      0.06      1946

    

 KEY FEATURES
✅ Auto Encoding — converts all columns to model-ready numeric values
✅ Two Models Trained — compare performance instantly
✅ AUC-ROC Metric — industry standard for credit scoring
✅ Fairness Audit — built-in gender parity check
✅ 300–850 Score Conversion — matches industry standard scale
✅ Grade Bands — A–E for easy interpretation
✅ Saved Model — ready to deploy in your API
✅ Single Applicant Scoring — drop-in ready for your dashboard